# Part 2 - 03 Fine-tuning: unfreeze the last layers of MobileNetV2

Starts from the model trained in notebook 02 (frozen base, trained head).

## Imports and configuration

In [ ]:
import os
import time

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight

from neurovision.data import CLASSES
from neurovision.preprocessing import prepare_data, make_dataset, encoder
from neurovision.transfer import IMAGE_SIZE, get_base, count_parameters, unfreeze_last_layers

RAW_DIR = "../data/raw"
TRANSFER_MODEL = "../models/transfer_best.keras"
UNFREEZE_LAST = 30
FINE_TUNE_LR = 1e-4
EPOCHS_FT = 1

## Load the data (same cleaning and split as before) and class weights from the training set

In [ ]:
data = prepare_data(RAW_DIR, size=IMAGE_SIZE, seed=42)
X_train, y_train, p_train = data["train"]
X_val, y_val, p_val = data["val"]

train_ds = make_dataset(X_train, y_train, training=True)
val_ds = make_dataset(X_val, y_val)

weights = compute_class_weight(
    "balanced", classes=np.arange(len(CLASSES)), y=encoder.transform(y_train)
)
class_weight = dict(enumerate(weights))
print(X_train.shape, X_val.shape)

## Load the transfer learning model and record its validation score (before fine-tuning)

In [ ]:
model = tf.keras.models.load_model(TRANSFER_MODEL)

val_loss_before, val_acc_before = model.evaluate(val_ds)
trainable, non_trainable = count_parameters(model)
print(f"Before: val loss {val_loss_before:.4f} | val accuracy {val_acc_before:.4f}")
print(f"Trainable: {trainable:,} | Non-trainable: {non_trainable:,}")

## Unfreeze the last layers of the base and recompile with a much lower learning rate

In [ ]:
model = unfreeze_last_layers(model, num_layers=UNFREEZE_LAST, learning_rate=FINE_TUNE_LR)

trainable, non_trainable = count_parameters(model)
print(f"Unfrozen layers: {UNFREEZE_LAST} | learning rate: {FINE_TUNE_LR}")
print(f"Trainable: {trainable:,} | Non-trainable: {non_trainable:,}")

## Callbacks: save the best model, stop early, resume after interruption

In [ ]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "../models/finetuned_best.keras", monitor="val_accuracy", save_best_only=True
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    ),
    tf.keras.callbacks.BackupAndRestore(backup_dir="../models/backup_finetune"),
]

## Retrain the head and the unfrozen layers together and measure the training time

In [ ]:
start = time.time()
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FT,
    class_weight=class_weight,
    callbacks=callbacks,
)
finetune_minutes = (time.time() - start) / 60
print(f"Fine-tuning time: {finetune_minutes:.1f} min for {len(history.history['loss'])} epoch(s)")

## Learning curves: accuracy and loss

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="validation")
axes[0].set_title("Accuracy (fine-tuning)")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="validation")
axes[1].set_title("Loss (fine-tuning)")
axes[1].set_xlabel("Epoch")
axes[1].legend()
plt.savefig("../reports/finetuning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Compare performance before and after fine-tuning (validation set, the test set stays untouched)

In [ ]:
val_loss_after, val_acc_after = model.evaluate(val_ds)

print(f"Before fine-tuning: val accuracy {val_acc_before:.4f} | val loss {val_loss_before:.4f}")
print(f"After fine-tuning:  val accuracy {val_acc_after:.4f} | val loss {val_loss_after:.4f}")

if val_acc_after > val_acc_before:
    print("Fine-tuning improved validation accuracy: keep models/finetuned_best.keras")
else:
    print("Fine-tuning did not improve validation accuracy: keep models/transfer_best.keras")